## Session 7, in class: from one household to many

Computational Macro (HWS 2026), University of Mannheim. Thursday, 22.10.2026, 10:00 to 12:00. Not graded.

This is the code behind the session 6 slides. Part 1 solves the household of the slides with the endogenous
grid method of week 5. Part 2 aggregates by Monte Carlo simulation, part 3 by Young's method, with a close
look at where the mass of one cell goes. Profiling the two, and one equilibrium wrapper for Huggett and
Aiyagari, follow in the coding session after the midterm (`Week_8`).

**How to work with this notebook**

- Run the cells from top to bottom (`Shift + Enter`). After a restart of the kernel, start again at the top:
  later cells use what earlier cells define.
- There are **4 gaps**, marked `___`. The comment line `# Gap k: ...` directly above a gap says what goes in.
  Every gap is one line, and most are one line of the slides.
- Cells without a gap run as they are. Read them anyway: they are part of the method, and the comments
  explain each step.
- Each part ends with a **Check**: a number to compare with, from the slides. Do not move on before it fits.
- **Slides.** "Session 6, slide 12" and the like refer to the session 6 slides, by the number at the bottom of
  each slide. Keep them open next to the notebook.
- The solution notebook is posted after the session.

**Plan**

| Part | Content | Gaps | Slides | Time |
| --- | --- | --- | --- | --- |
| 1 | The household of the slides, solved with the endogenous grid method (given) | | S6: 2 | 5 min |
| 2 | Aggregation by Monte Carlo simulation: a panel of households, and the sampling error | | S6: 6 to 10 | 10 min |
| 3 | Young's method: the lottery, where the mass of one cell goes, the policy chain, the stationary distribution | 1 to 4 | S6: 11 to 17 | 30 min |

**Notation, slides to code**

| Slides | Meaning | Code |
| --- | --- | --- |
| $\mathcal{A}$, $\mathcal{Z}$, $\boldsymbol{\Pi}$ | asset grid, income grid, transition matrix (rows sum to one) | `gri.a`, `gri.z`, `gri.Pi` |
| $\boldsymbol{\lambda}^z$ | stationary distribution of the income chain | `gri.lam_z` |
| $\mathbf{c}$, $\mathbf{g}$, $a'_{im} = g(a_i, z_m)$ | consumption and savings policy on the grid | `C`, `G` |
| $l$, $w_{im} = (a'_{im} - a_l) / (a_{l+1} - a_l)$ | the lottery: left neighbour of $a'_{im}$ and the weight on the right one | `l`, `w` |
| $\mathbf{Q}_{\mathbf{g}}$ | the policy chain, $N_a N_z \times N_a N_z$, row $(i, m)$ is the state $(a_i, z_m)$ | `Q` |
| $\boldsymbol{\lambda}$, $\boldsymbol{\lambda} = \mathbf{Q}_{\mathbf{g}}^\top \boldsymbol{\lambda}$ | distribution over $(a, z)$, stationary | `lam` |
| $A = \boldsymbol{\lambda}^\top \mathbf{a}$ | mean assets | `mean_assets(lam, gri)` |

**Toolbox: the few tools this notebook uses that are new**

| Tool | What it does | Example |
| --- | --- | --- |
| `searchsortedlast(grid, x)` | the index `l` of the grid point just below (or at) `x` (for a sorted `grid`) | locating the lottery |
| `clamp(x, lo, hi)` | `x`, but not below `lo` and not above `hi` | keep `l` inside `1, ..., N_a - 1` |
| `reshape(1:n, na, nz)` | the numbers `1, ..., n` as an `na` by `nz` table | `idx[i, m]` is the row of state $(a_i, z_m)$ in `Q` |
| `sparse(rows, cols, vals, n, n)` | a sparse matrix from triplets: `vals[k]` sits at `(rows[k], cols[k])` | $\mathbf{Q}_{\mathbf{g}}$ with $2 N_z$ entries per row |
| `Q' * lam` | sparse matrix times vector | one step of the distribution |
| `rng = MersenneTwister(seed)`, `rand(rng, N)` | a seeded random number generator, `N` uniforms | the simulation |

Everything else is from the earlier notebooks: `A * B`, the dots `.+`, `.*`, `interp1`, `sum(A, dims = 2)`.
A policy is stored as an $N_a \times N_z$ matrix: row `i` is the asset grid point $a_i$, column `m` the income state $z_m$.

### 0. Packages

Nothing new to install: `SparseArrays`, `Random`, `Statistics`, and `Profile` ship with Julia, and `QuantEcon`
(for the income chain) is in the course environment since week 5. The first cell activates the course
environment in `Julia/` and installs every package listed there that is missing on your laptop.

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # the course environment in Julia/
Pkg.instantiate()                        # installs every package it lists that is missing on your laptop

  Activating project at `c:\Users\manuc\OneDrive\Dokumente\GitHub\Manucher-Karimi-Computational_Macro-2026\Julia`


Pkg.Types.PkgError: Could not parse manifest: TOML Parser error:
C:\Users\manuc\OneDrive\Dokumente\GitHub\Manucher-Karimi-Computational_Macro-2026\Julia\Manifest.toml:5:1 error: invalid bare key character: '<'
  <<<<<<< Updated upstream
  ^                         

In [ ]:
using Plots
using Printf
using LinearAlgebra
using SparseArrays                                        # sparse matrices, for the policy chain
using Random
using Statistics
using QuantEcon                                           # rouwenhorst, as in week 5

### 1. The household of the slides (given)

*Session 6, slides 2 and 4.*

The household of sessions 4 and 5 with CRRA utility, $u(c) = c^{1 - \gamma} / (1 - \gamma)$, $\gamma = 2$,
$\beta = 0.96$, the income chain of session 4 ($\ln z' = \rho \ln z + \sigma_\varepsilon \varepsilon'$ with
$\rho = 0.9$, $\sigma_\varepsilon = 0.2$, on $N_z = 7$ Rouwenhorst states, $z = e^{\ln z}$), and the budget
constraint $c + a' = w z + R a$ with $a' \geq \underline{a}$. In parts 2 and 3 prices are given, $r = 0.03$ and
$w = 1$, and $\underline{a} = 0$. Huggett changes $\underline{a}$, Aiyagari changes $w$, both change $r$, so
the solver takes `r`, `w`, and the grids as arguments.

The asset grid has $N_a = 200$ points on $[\underline{a}, 100]$, **quadratically spaced**: dense near the
limit, where the policy bends, sparse at the top, where it is almost linear. The household problem is solved
with the endogenous grid method of session 5 (linear interpolation, policy tolerance $10^{-8}$). The only
changes to week 5's `egm_step` are CRRA marginal utility, income $w z_m$ in cash on hand, and the persistent
chain in $\mathbf{EMU} = u_c(\mathbf{c}) \boldsymbol{\Pi}^\top$, which was there already.

In [ ]:
Base.@kwdef struct EconomicParameters
    γ::Float64 = 2.0          # relative risk aversion
    β::Float64 = 0.96         # discount factor
    ρ::Float64 = 0.9          # persistence of ln z
    sig_eps::Float64 = 0.2    # standard deviation of the innovation to ln z
end

Base.@kwdef struct NumericalParameters          # the numerical choices: grid sizes, tolerances
    na::Int = 200             # asset grid points N_a
    nz::Int = 7               # income grid points N_z
    amax::Float64 = 100.0     # upper grid bound a_{N_a}
    crit_pol::Float64 = 1e-8  # the EGM stops when the policy moves by less
    crit_dist::Float64 = 1e-12  # the distribution iteration stops when lam moves by less (part 3)
    maxiter::Int = 100_000    # iteration cap, the safety net
end

In [ ]:
struct Grids                                    # a container for the grids, filled by make_grids
    a::Vector{Float64}        # asset grid, N_a
    z::Vector{Float64}        # income grid, N_z
    Pi::Matrix{Float64}       # transition matrix, N_z x N_z
    lam_z::Vector{Float64}    # stationary distribution of the income chain, N_z
end

function make_grids(amin, par, mpar)
    # The income chain of session 4 on N_z Rouwenhorst states (week 5, part 1), income levels z = exp(ln z),
    # and the quadratically spaced asset grid: a_i = amin + (amax - amin) s_i^2 with s_i equidistant on [0, 1]
    mc = rouwenhorst(mpar.nz, par.ρ, par.sig_eps)        # the chain for ln z, as in week 5
    z = exp.(mc.state_values)                            # income levels
    lam_z = vec(stationary_distributions(mc)[1])         # the stationary distribution of the chain, week 4
    s = collect(range(0.0, 1.0, length = mpar.na))       # N_a equidistant points on [0, 1]
    a = amin .+ (mpar.amax - amin) .* s .^ 2             # squared: dense near amin, sparse near amax
    return Grids(a, z, Matrix(mc.p), lam_z)
end

In [ ]:
par, mpar = EconomicParameters(), NumericalParameters()  # the default values listed above
gri = make_grids(0.0, par, mpar)                         # borrowing limit a >= 0 for parts 2 and 3
println("income states: ", round.(gri.z, digits = 3))
println("stationary distribution of income: ", round.(gri.lam_z, digits = 3), "  mean income E[z] = ", round(gri.lam_z' * gri.z, digits = 4))
println("asset grid: first spacing ", round(gri.a[2] - gri.a[1], digits = 4), "  last spacing ", round(gri.a[end] - gri.a[end-1], digits = 3))

**The solver.** `uprime` and `uprime_inv` are the CRRA versions of week 5's CARA functions.
`egm_step` and `solve_household` are week 5's `egm_step` and `egm` with `r`, `w`, and the grids as arguments.
The guess is to consume income and interest, $c_0 = r a + w z$, unless a better guess is passed in `C0`
(next week explains why that matters). The solver returns consumption `C`, the savings policy `G` with
$a'_{im} = g(a_i, z_m) = w z_m + R a_i - c_{im}$, and the iteration count.

In [ ]:
# The dots inside make both functions work elementwise on a number, a vector, or a matrix
uprime(c, par) = c .^ (-par.γ)                  # marginal utility of CRRA utility u(c) = c^(1 - γ) / (1 - γ)
uprime_inv(m, par) = m .^ (-1 / par.γ)

In [ ]:
function interp1(xq, xgrid, fgrid)
    # Linear interpolation of the points (xgrid, fgrid) at every query point in xq, flat outside the grid (week 5)
    out = similar(xq)
    for (k, x) in enumerate(xq)
        i = clamp(searchsortedlast(xgrid, x), 1, length(xgrid) - 1)   # the interval [xgrid[i], xgrid[i+1]] around x
        w = clamp((x - xgrid[i]) / (xgrid[i+1] - xgrid[i]), 0.0, 1.0) # weight on the right end point, flat outside
        out[k] = (1 - w) * fgrid[i] + w * fgrid[i+1]
    end
    return out
end

In [ ]:
function egm_step(C, r, w, par, gri)
    # One step of the endogenous grid method (session 5, slide 20): from the policy C to the new policy on the grid
    R = 1 + r
    EMU = uprime(C, par) * gri.Pi'              # expected marginal utility tomorrow at every (a'_l, z_m), week 5 gap 5
    ctilde = uprime_inv(par.β * R .* EMU, par)  # (a) consumption today from the inverted Euler equation
    atilde = (ctilde .+ gri.a .- w .* gri.z') ./ R   # (b) today's assets from the budget constraint
    Cnew = similar(C)                           # (c) back to the grid, one income state at a time
    for m in 1:length(gri.z)
        Cnew[:, m] = interp1(gri.a, atilde[:, m], ctilde[:, m])
        below = gri.a .< atilde[1, m]           # poorer than the household that saves a' = a_1: the limit binds
        Cnew[below, m] = w * gri.z[m] .+ R .* gri.a[below] .- gri.a[1]   # consume everything down to a' = a_1
    end
    return Cnew
end

In [ ]:
function solve_household(r, w, par, mpar, gri; C0 = nothing)
    # The EGM loop of week 5: iterate egm_step until the policy stops moving. Returns C, the savings policy G, and the count.
    # C0: an optional starting guess for the policy (the solution at nearby prices, next week)
    R = 1 + r
    C = C0 === nothing ? max.(r .* gri.a .+ w .* gri.z', 1e-6) : copy(C0)   # the guess: consume interest and income
    it = 0
    while it < mpar.maxiter
        it += 1
        Cnew = egm_step(C, r, w, par, gri)
        dist = maximum(abs.(Cnew .- C))         # largest change of the policy
        C = Cnew
        dist < mpar.crit_pol && break
    end
    G = w .* gri.z' .+ R .* gri.a .- C          # savings a' = w z + R a - c, N_a x N_z
    return C, G, it
end

In [ ]:
r0, w0 = 0.03, 1.0                              # the prices of parts 2 and 3
t0 = time()
C0, G0, it0 = solve_household(r0, w0, par, mpar, gri)
@printf("EGM: %d iterations, %.3f s\n", it0, time() - t0)
println("net saving g(a, z) - a at a = 0, by income state: ", round.(G0[1, :] .- gri.a[1], digits = 3))
i10 = argmin(abs.(gri.a .- 10))                 # the grid point closest to a = 10
println("net saving at a = ", round(gri.a[i10], digits = 2), ", by income state: ", round.(G0[i10, :] .- gri.a[i10], digits = 3))

**Check (session 6, slide 2):** mean income $E[z] = 1.11$, EGM converges in $369$ iterations. At $a = 0$ the two
lowest income states stay at the limit (net saving $0$), the highest saves about $0.9$. At $a = 10$ every state
but the two highest dissaves. A household with median income has a target wealth where saving turns negative:
that is where mass will pile up.

In [ ]:
# Nothing to fill in: the savings policy as net saving g(a, z) - a, as on session 6, slide 2
sel = gri.a .<= 30
p = plot(xlabel = "assets today a", ylabel = "net saving g(a, z) - a", title = "The savings policy", titlefontsize = 10, size = (600, 340))
for (k, lab) in ((1, "lowest income z_1"), (4, "median income z_4"), (7, "highest income z_7"))
    plot!(p, gri.a[sel], G0[sel, k] .- gri.a[sel], linewidth = 1.5, label = lab)
end
hline!(p, [0.0], color = :black, linestyle = :dot, label = "")

### 2. Aggregation by Monte Carlo simulation (nothing to fill in)

*Session 6, slides 6 to 10.*

The direct way to a distribution: $N$ households, each with its own assets and income, moved forward $T$
periods with the policy and the income chain. Slide 6: start everybody at $a = 0$ with income drawn from
$\boldsymbol{\lambda}^z$, and in every period (i) read tomorrow's assets off the policy, interpolated between
grid points, and (ii) draw tomorrow's income from the row of $\boldsymbol{\Pi}$ of today's income, with the
cumulated row and one uniform (week 5, part 1). **Vectorize over households, not over time:** one
interpolation per income state per period, one array of $N$ uniforms.

The cross section after $T$ periods is a sample from the stationary distribution, once the burn in is over.

In [ ]:
function simulate_panel(G, gri, N, T, seed)
    # N households for T periods. Returns the final cross section (assets, income index) and the mean of assets in every period
    rng = MersenneTwister(seed)                 # seeded, so that the run is reproducible
    cum = cumsum(gri.Pi, dims = 2)
    cum_z = cumsum(gri.lam_z)                   # cumulated stationary distribution, for the initial income
    a = fill(gri.a[1], N)                       # everybody starts at the limit
    m = [searchsortedfirst(cum_z, u) for u in rand(rng, N)]   # income indices drawn from the stationary distribution
    means = zeros(T)                            # mean assets in every period
    for t in 1:T
        anew = similar(a)
        for k in 1:length(gri.z)                # (i) tomorrow's assets: one interpolation per income state
            sel = m .== k                       # the households with income z_k today
            anew[sel] = interp1(a[sel], gri.a, G[:, k])
        end
        a = anew
        u = rand(rng, N)                        # (ii) tomorrow's income: one uniform per household
        m = [min(searchsortedfirst(view(cum, m[j], :), u[j]), length(gri.z)) for j in 1:N]   # the first n with F_mn >= u (week 5 by hand)
        means[t] = mean(a)
    end
    return a, m, means
end

In [ ]:
N, T = 10_000, 1_000
t0 = time()
a_sim, m_sim, means = simulate_panel(G0, gri, N, T, 11)
t_sim = time() - t0
@printf("simulation of N = %d households for T = %d periods: %.2f s\n", N, T, t_sim)
@printf("mean assets %.3f, share at the limit %.3f, median %.2f\n", mean(a_sim), mean(a_sim .<= gri.a[1] + 1e-10), median(a_sim))
println("mean assets over time, t = 10, 50, 100, 300, 1000: ", round.(means[[10, 50, 100, 300, 1000]], digits = 2))

**Check (session 6, slide 10):** mean assets about $8.5$ (the seed decides the second decimal), $1$ to $2$ percent
at the limit, a median around $6$. The mean needs about $300$ periods to settle (the burn in), the first $10$
periods are still close to zero.

**The sampling error.** Repeat the simulation with three seeds for $N = 1{,}000$ and $10{,}000$ households
($T = 500$ is enough after the burn in check above). The standard deviation of the mean across seeds falls
like $1 / \sqrt{N}$: one more digit costs a hundred times the households, and the time grows with them.
(Next week the numbers go next to Young's method.)

In [ ]:
mc_rows = []                                    # (N, mean over seeds, sd across seeds, seconds per simulation), for next week
for Nn in (1_000, 10_000)
    ms = Float64[]
    tt = time()
    for seed in 0:2
        a_s, _, _ = simulate_panel(G0, gri, Nn, 500, seed)
        push!(ms, mean(a_s))
    end
    tt = (time() - tt) / 3
    push!(mc_rows, (Nn, mean(ms), std(ms, corrected = false), tt))
    @printf("N = %7d: mean assets over 3 seeds %.3f, sd across seeds %.3f, %.2f s per simulation\n", Nn, mean(ms), std(ms, corrected = false), tt)
end

**Check:** the standard deviation across seeds is about $0.2$ and $0.07$ (with three seeds the first digit
moves between runs): it falls by a factor of about $\sqrt{10} \approx 3.2$ for the factor of $10$ in $N$. To pin the mean to
two decimals you would need $N = 10^5$ and a hundred times the time. Keep these numbers: Young's method delivers
the mean to twelve digits in part 3, in a fraction of the time.

### 3. Young's method (gaps 1 to 4)

*Session 6, slides 11 to 17.*

**The policy turns the state into a Markov chain (slide 11).** A household at $(a_i, z_m)$ saves
$a'_{im} = g(a_i, z_m)$ and draws $z_n$ with probability $\pi_{mn}$. If $a'_{im}$ were a grid point $a_{G_{im}}$,
the pair $(a, z)$ would be a Markov chain with $N_a N_z$ states, and row $(i, m)$ of its transition matrix would
have $\pi_{mn}$ in column $(G_{im}, n)$. The distribution then moves forward exactly as the income distribution
did on slide 3, $\boldsymbol{\lambda}_{t+1} = \mathbf{Q}_{\mathbf{g}}^\top \boldsymbol{\lambda}_t$, and the stationary
distribution solves $\boldsymbol{\lambda} = \mathbf{Q}_{\mathbf{g}}^\top \boldsymbol{\lambda}$.

**The lottery (slide 12).** The EGM policy lies between grid points, $a'_{im} \in [a_l, a_{l+1}]$. Young (2010)
splits the mass with the linear interpolation weights: $1 - w_{im}$ to $a_l$ and $w_{im}$ to $a_{l+1}$, with

$$
w_{im} = \frac{a'_{im} - a_l}{a_{l+1} - a_l},
$$

a lottery over two grid points whose mean is exactly $a'_{im}$. Rounding to the nearest grid point would create
absorbing states (the fixed points around $k^*$ in problem set 1). The lottery lets mass flow.

`lottery` locates $l$ for every $(i, m)$ at once (`np.searchsorted` on the whole array, kept inside
$0, \ldots, N_a - 2$ so that $l + 1$ exists) and computes the weights.

In [ ]:
function lottery(G, gri)
    # For every policy value a'_im: the index l of the grid point just below it, and the weight w_im on the point above
    na = length(gri.a)
    l = clamp.(searchsortedlast.(Ref(gri.a), G), 1, na - 1)   # locate: a_l <= a'_im < a_{l+1}, l at most N_a - 1
    # Gap 1: the weight on a_{l+1}: the distance of G from a_l, divided by the spacing a_{l+1} - a_l (gri.a[l]
    #        is an N_a x N_z matrix, like G, use the dots)
    w = ___
    w = clamp.(w, 0.0, 1.0)                     # exactly at the top of the grid, w would be 1 + 1e-16
    return l, w
end

In [ ]:
l0, w0_ = lottery(G0, gri)
println("weights inside [0, 1]: ", all(0 .<= w0_ .<= 1))
# The lottery reproduces the policy, except where the richest households at a = 100 would save beyond the grid
# (two points; the lottery caps them at a_N, and part 3 checks that no mass ever gets there)
println("lottery mean equals the policy (capped at a_N): ", isapprox((1 .- w0_) .* gri.a[l0] .+ w0_ .* gri.a[l0 .+ 1], min.(G0, gri.a[end])))
println("policy values above the top of the grid: ", sum(G0 .> gri.a[end]))
i17 = argmin(abs.(gri.a .- 17.0))               # the example of slide 12: a = 17.0, income z_6
@printf("a = %.2f, z_6: a' = %.2f lies between %.2f and %.2f, w = %.2f\n", gri.a[i17], G0[i17, 6], gri.a[l0[i17, 6]], gri.a[l0[i17, 6] + 1], w0_[i17, 6])

**Check (slide 12):** at $a = 17.0$ with income $z_6$ the household saves $a' = 17.56$, between the grid points
$17.40$ and $17.82$, so $w = 0.40$. Two policy values sit above $a_N = 100$ (the two richest income states at the
top of the grid). The lottery caps them at $a_N$, which is harmless as long as no mass arrives there (check 3 below).

**Where the mass of one cell goes (slide 13, gap 2).** Take the cell $(a_i, z_m)$ of the example. Its mass
splits $2 N_z$ ways: for every income state $z_n$ tomorrow, a share $\pi_{mn} (1 - w_{im})$ lands on $(a_l, z_n)$
and a share $\pi_{mn} w_{im}$ on $(a_{l+1}, z_n)$. These $2 N_z$ numbers are the nonzero entries of row $(i, m)$ of
$\mathbf{Q}_{\mathbf{g}}$. Two things to check by hand: the shares add up to one (all the mass goes somewhere),
and the mean of tomorrow's assets under these shares is the policy value $a'_{im}$ itself (the lottery is fair).

In [ ]:
# The row of Q for the household of the example, written out: destination, share, as on slide 13
i, m = i17, 6                                   # the cell (a_i, z_m) = (17.0, z_6)
l_im, w_im = l0[i, m], w0_[i, m]
rows = Tuple{Float64, Float64, Float64}[]
for n in 1:length(gri.z)                        # income tomorrow
    # Gap 2: the share landing on (a_l, z_n): the income probability pi_mn times the lottery weight on the
    #        left neighbour, 1 - w_im
    push!(rows, (gri.a[l_im], gri.z[n], ___))
    push!(rows, (gri.a[l_im + 1], gri.z[n], gri.Pi[m, n] * w_im))   # the share landing on (a_{l+1}, z_n)
end
@printf("mass leaving (a, z) = (%.2f, z_6), policy a' = %.2f, lottery between %.2f and %.2f with w = %.2f\n", gri.a[i], G0[i, m], gri.a[l_im], gri.a[l_im + 1], w_im)
println("   a' tomorrow   z tomorrow   share")
for (a_to, z_to, share) in rows
    share > 1e-4 && @printf("   %9.2f   %9.2f   %.4f\n", a_to, z_to, share)   # skip the tiny shares of far income states
end
@printf("shares add up to %.6f, mean of a' tomorrow %.4f (the policy says %.4f)\n", sum(s for (_, _, s) in rows), sum(a_to * s for (a_to, _, s) in rows), G0[i, m])

**Check (slide 13):** the largest shares go to $z_6$ tomorrow ($\pi_{66} = 0.73$ split $0.44$ and $0.29$), the
shares add up to one, and the mean of $a'$ tomorrow is $17.56$, the policy value. That is the whole of Young's
method, applied to one cell. The policy chain does it for all $1{,}400$ cells at once.

**The policy chain (slides 13 to 15, gap 3).** $\mathbf{Q}_{\mathbf{g}}$ has $N_a N_z = 1{,}400$ rows and columns but
only $2 N_z = 14$ nonzero entries per row, the shares of the previous cell: $\pi_{mn} (1 - w_{im})$ in column
$(l, n)$ and $\pi_{mn} w_{im}$ in column $(l + 1, n)$, for $n = 1, \ldots, N_z$. So it is built as a **sparse
matrix from triplets** (row, column, value), $19{,}600$ entries instead of $1{,}400^2 \approx 2$ million.

States are numbered by the table `idx`: `idx[i, m]` is the row (and column) of the state $(a_i, z_m)$, in the
order of Howard's step of week 5, $(i, m) \mapsto i + (m - 1) N_a$ (assets count fastest). The order is a
convention, but it must be the same for rows, columns, and the `reshape` that turns the vector
$\boldsymbol{\lambda}$ back into an $N_a \times N_z$ table. The loop runs over the income states $m$ today and $n$
tomorrow. Inside, all $N_a$ asset points are handled at once.

In [ ]:
function transition_matrix(G, gri)
    # Q[(i, m), (l, n)] = Pi[m, n] (1 - w_im) and Q[(i, m), (l + 1, n)] = Pi[m, n] w_im: the policy chain as a sparse matrix
    na, nz = length(gri.a), length(gri.z)
    idx = reshape(1:(na * nz), na, nz)          # idx[i, m] = i + (m - 1) N_a: the row of state (a_i, z_m), the order of Howard's step (week 5)
    l, w = lottery(G, gri)                      # the lottery of every (i, m)
    rows, cols, vals = Int[], Int[], Float64[]  # the triplets, collected column block by column block
    for m in 1:nz                               # income today
        for n in 1:nz                           # income tomorrow
            # mass 1 - w_im goes to (a_l, z_n) ...
            append!(rows, idx[:, m]); append!(cols, idx[l[:, m], n]); append!(vals, gri.Pi[m, n] .* (1 .- w[:, m]))
            # ... and mass w_im to (a_{l+1}, z_n)
            # Gap 3: mirror the line above: the column of state (a_{l+1}, z_n), and the mass pi_mn w_im
            append!(rows, idx[:, m]); append!(cols, ___); append!(vals, ___)
        end
    end
    return sparse(rows, cols, vals, na * nz, na * nz)
end

In [ ]:
t0 = time()
Q0 = transition_matrix(G0, gri)
@printf("Q built in %.1f ms: size %s, %d stored entries, %.2f percent of the matrix\n", 1000 * (time() - t0), size(Q0), nnz(Q0), 100 * nnz(Q0) / size(Q0, 1)^2)
rowsums = vec(sum(Q0, dims = 2))
@printf("rows sum to one: largest deviation %.1e\n", maximum(abs.(rowsums .- 1)))
idx0 = reshape(1:(mpar.na * mpar.nz), mpar.na, mpar.nz)   # the same table, for looking up rows below
row = Vector(Q0[idx0[i17, 6], :])               # the row of the example cell, as a dense vector
@printf("row of (17.0, z_6): %d nonzero entries, largest %.4f, sum %.6f\n", count(!iszero, row), maximum(row), sum(row))

**Check (slides 13 and 15):** $1{,}400 \times 1{,}400$, about $19{,}600$ stored entries ($1$ percent of the matrix;
a few entries are stored zeros where $w_{im} = 0$ exactly, at the limit), every row sums to one up to rounding,
and the row of the example cell has the $14$ entries of the previous cell, the largest $0.44$. Note on gap 3: the
hint names the two gaps of that line together, since one is the mirror of the other.

**The stationary distribution (slides 14 and 16, gap 4).** Row $(i, m)$ says where the mass at $(a_i, z_m)$
goes. Tomorrow's mass at a cell $(l, n)$ is what **arrives** there, from every cell that sends some: a column sum
weighted by today's distribution, $\lambda_{ln, t+1} = \sum_{(i, m)} Q_{(i, m), (l, n)} \lambda_{im, t}$, in one
line $\boldsymbol{\lambda}_{t+1} = \mathbf{Q}_{\mathbf{g}}^\top \boldsymbol{\lambda}_t$ (slide 14). Distributions move
with the **transpose**. `Q @ lam` would take expectations, the Howard step of week 5, and converge to a constant
vector. Start from the uniform distribution, or from a distribution passed in `lam0` (next week), and iterate until
the largest change is below $10^{-12}$.

In [ ]:
function stationary_dist(Q, mpar, gri; lam0 = nothing)
    # Iterate lam_{n+1} = Q' lam_n from the uniform distribution (or from lam0). Returns lam as an N_a x N_z table, and the count
    n = size(Q, 1)
    lam = lam0 === nothing ? fill(1 / n, n) : vec(lam0)   # the start, as a vector in the order of idx
    QT = sparse(Q')                             # the transpose once, stored as its own sparse matrix
    it = 0
    while it < mpar.maxiter
        it += 1
        # Gap 4: one step forward: the transpose of Q times the current distribution
        lam_new = ___
        dist = maximum(abs.(lam_new .- lam))    # largest change
        lam = lam_new
        dist < mpar.crit_dist && break
    end
    return reshape(lam, length(gri.a), length(gri.z)), it   # back to the (a, z) table, the same order as idx
end

In [ ]:
t0 = time()
lam0, itl = stationary_dist(Q0, mpar, gri)
t_young = time() - t0
@printf("Young's iteration: %d iterations to 1e-12, %.0f ms\n", itl, 1000 * t_young)
# One step by hand, for one cell: the mass arriving at the example cell (17.0, z_6) is the weighted column sum of slide 14
lam_vec = vec(lam0)                             # lam as a vector in the order of idx
col = Vector(Q0[:, idx0[i17, 6]])               # column (17.0, z_6) of Q: who sends mass there, and how much
@printf("mass arriving at (17.0, z_6): %.3e from %d cells, against lam there %.3e (equal: stationary)\n", col' * lam_vec, count(!iszero, col), lam0[i17, 6])

**Aggregates and checks (slide 16).** Mean assets are $A = \boldsymbol{\lambda}^\top \mathbf{a} = \sum_{i, m} \lambda_{im} a_i$.
Then three checks before you trust a distribution: mass one, the marginal over assets equals
$\boldsymbol{\lambda}^z$ (a cheap test of the column index), and nothing at the top of the grid. The simulation
of part 2 is the fourth, external check.

In [ ]:
function mean_assets(lam, gri)
    # A = Σ_{i, m} lam_im a_i, with lam as an N_a x N_z table
    return sum(lam .* gri.a)
end

In [ ]:
A0 = mean_assets(lam0, gri)
lam_a0 = vec(sum(lam0, dims = 2))               # the marginal distribution of assets
@printf("mean assets %.4f (simulation: %.4f)\n", A0, mean(a_sim))
@printf("share at the limit %.4f (simulation: %.4f)\n", sum(lam0[1, :]), mean(a_sim .<= gri.a[1] + 1e-10))
@printf("check 1, mass: %.12f\n", sum(lam0))
@printf("check 2, marginal over assets against lam_z: %.1e\n", maximum(abs.(vec(sum(lam0, dims = 1)) .- gri.lam_z)))
@printf("check 3, mass at the top of the grid: %.1e\n", sum(lam0[end, :]))
println("mean assets by income state: ", round.(vec(sum(lam0 .* gri.a, dims = 1)) ./ vec(sum(lam0, dims = 1)), digits = 1))
cdf0 = cumsum(lam_a0)
@printf("median %.1f, share below one year of mean income: %.2f\n", gri.a[searchsortedfirst(cdf0, 0.5)], cdf0[searchsortedlast(gri.a, gri.lam_z' * gri.z)])

**Check (slides 16 and 17):** $1{,}267$ iterations in a few tens of milliseconds, mean assets $8.517$ against about
$8.50$ in the simulation, $1.6$ percent at the limit, mass $1$ to twelve digits, the marginal over assets equal
to $\boldsymbol{\lambda}^z$ to $10^{-13}$, $10^{-9}$ at $a = 100$. Mean assets by income state rise from $4.2$ at
$z_1$ to $17.3$ at $z_7$: the precautionary motive at work. Median $6.3$, $14$ percent below one year of mean income.

In [ ]:
# Nothing to fill in: the simulated cross section against Young's distribution (session 6, slide 17).
# Young's mass per grid point is turned into a density by dividing by the width of the cell around each point.
function density(lam_a, agrid)
    mids = [agrid[1]; (agrid[1:end-1] .+ agrid[2:end]) ./ 2; agrid[end]]   # cell boundaries: midpoints between neighbours
    return lam_a ./ diff(mids)                  # mass per unit of assets
end

edges = range(0, 30, length = 61)
inside = a_sim .> gri.a[1] + 1e-10              # the histogram without the spike at the limit
p1 = histogram(a_sim[inside], bins = edges, weights = fill(1 / N / step(edges), sum(inside)), alpha = 0.6,
               label = "simulated cross section, N = $N, T = $T", xlabel = "assets a", ylabel = "density", xlims = (0, 30),
               title = "Same distribution", titlefontsize = 10)
plot!(p1, gri.a[2:end], density(lam_a0, gri.a)[2:end], color = :black, linewidth = 1.2, label = "Young's method, mass per unit of assets")
p2 = plot(1:T, means, linewidth = 1.2, label = "mean assets in the simulation", xscale = :log10, xlabel = "period t", ylabel = "mean assets",
          title = "The burn in", titlefontsize = 10, legend = :bottomright)
hline!(p2, [A0], color = :black, linestyle = :dash, label = "Young's method")
plot(p1, p2, layout = (1, 2), size = (950, 340))

**To discuss (slide 17).** The histogram sits on Young's density, and the two means differ by less than one
standard error of the simulation. What Young's method assumes: mass sits on grid points (the grid sets the
resolution), the mean is exact, the variance is slightly too large by the lottery's spread. What it needs: a grid
for the endogenous state and a finite chain for the exogenous one. Simulate anyway when the state space is too
large for the matrix, when the question is about individual histories, or once, as a check of a new
$\mathbf{Q}_{\mathbf{g}}$, as here.